# 01 Data exploration

What is in BongoScam and the Chichewa set, how cleaning changes them, and the shortcut that makes BongoScam easy.

Regenerate the processed files with `python -m src.data`.

In [1]:
import sys, os, json
sys.path.insert(0, os.path.abspath('..'))
os.environ.setdefault('MPLCONFIGDIR', os.path.abspath('../.cache/mpl'))
import pandas as pd
from IPython.display import Image, Markdown, display
pd.set_option('display.max_colwidth', 140)
pd.set_option('display.width', 200)
from src import config
S = pd.read_csv(config.RESULTS / 'experiments.csv')          # mean (and sd) over seeds
BY_SEED = pd.read_csv(config.RESULTS / 'experiments_by_seed.csv')

def table(models, sets, variant='clean', split='template', metric='f1'):
    t = S[S.model.isin(models) & S.set.isin(sets) & (S.variant == variant) & (S.split == split)]
    return t.pivot_table(index='model', columns='set', values=metric).reindex(models).round(3)

In [2]:
stats = json.loads((config.RESULTS / 'data_stats.json').read_text())
{k: v for k, v in stats.items() if not k.startswith('split')}

{'bongo_raw_rows': 1508,
 'bongo_raw_labels': {'scam': 1000, 'trust': 508},
 'bongo_exact_duplicates_dropped': 131,
 'bongo_labels_after_exact_dedup': {'scam': 880, 'not scam': 497},
 'bongo_identical_after_masking_dropped': 313,
 'bongo_identical_after_masking_by_label': {'scam': 313},
 'bongo_rows': 1064,
 'bongo_labels': {'scam': 567, 'not scam': 497},
 'bongo_words_median': 11.0,
 'share_with_phone_or_link': {'scam': 0.8589065255731922, 'not scam': 0.0},
 'phone_rule_f1_all_messages': 0.9240986717267552,
 'words_min_scam': 8,
 'words_median': {'scam': 12.0, 'not scam': 7.0},
 'bongo_templates': 909,
 'bongo_templates_with_2plus': 73,
 'bongo_messages_in_shared_templates': 228,
 'bongo_largest_template': 31,
 'bongo_scam_messages_in_shared_templates': 226,
 'chichewa_rows_raw': 824,
 'chichewa_by_source': {'D_CHI': {'fraud': 338, 'normal': 338},
  'telcoSMS_CHI': {'normal': 148}},
 'chichewa_exact_duplicates': 78,
 'chichewa_identical_after_masking_dropped': 91,
 'chichewa_rows': 73

## Raw BongoScam: labels and duplicates

The raw file has 1,508 rows; 131 are exact copies. After masking numbers, amounts and links, another 313 scams become identical to an earlier message (same script, different number), so they are dropped too.

In [3]:
from src.data import load_bongo_raw, load_bongo, load_chichewa
raw = pd.read_csv(config.BONGO_CSV)
print(raw.Category.value_counts(), '\nexact duplicates:', raw.Sms.duplicated().sum())
bongo = load_bongo()
bongo.label.map(config.LABEL_NAMES).value_counts()

Category
scam     1000
trust     508
Name: count, dtype: int64 
exact duplicates: 131


label
scam        567
not scam    497
Name: count, dtype: int64

## What masking does

In [4]:
from src.preprocess import preprocess
for t in load_bongo_raw().raw_text.sample(6, random_state=3):
    print('RAW :', t[:120]); print('SEEN:', preprocess(t)[:120]); print()

RAW : Hali ya afya yako ikoje sasa?
SEEN: Hali ya afya yako ikoje sasa?

RAW : Naomba unitumie iyo Hela kwenye namba hii ya Halotel 0789723456. jina (PEREGIA FILIPO)
SEEN: Naomba unitumie iyo Hela kwenye namba hii ya Halotel <PHONE> . jina (PEREGIA FILIPO)

RAW : FRIIMASON TIMIZA NDOTO KUMILIKI PESA MAJUMBA NA MAGARI YA KIFAHARI PIA KUZA BIASHARA VIPAJI NYOTA NA PETE. TUPIGIE SIMU 
SEEN: FRIIMASON TIMIZA NDOTO KUMILIKI PESA MAJUMBA NA MAGARI YA KIFAHARI PIA KUZA BIASHARA VIPAJI NYOTA NA PETE. TUPIGIE SIMU 

RAW : Yanga wana mashabiki wa kweli. Walijitokeza kwa wingi jana uwanjani.
SEEN: Yanga wana mashabiki wa kweli. Walijitokeza kwa wingi jana uwanjani.

RAW : Iyo ela tuma humu kwenye Airtel 0788542784 Jina lije KHADIJA SUMBAI.
SEEN: Iyo ela tuma humu kwenye Airtel <PHONE> Jina lije KHADIJA SUMBAI.

RAW : Utanitumia kwenye ii 0689592818 airtel jina RAMATUNGU namba yangu inadeni usiitumie
SEEN: Utanitumia kwenye ii <PHONE> airtel jina RAMATUNGU namba yangu inadeni usiitumie



## Message length

In [5]:
bongo['words'] = bongo.text.str.split().str.len()
bongo.groupby(bongo.label.map(config.LABEL_NAMES)).words.describe().round(1)

,count,mean,std,min,25%,50%,75%,max
label,,,,,,,,
not scam,497.0,7.9,3.7,1.0,5.0,7.0,10.0,27.0
scam,567.0,14.4,5.8,8.0,10.0,12.0,18.0,46.0


## Templates: the same script sent many times

Two messages share a template when their masked, lower-cased texts share at least 80% of character 5-grams. The template-disjoint split keeps each template on one side.

In [6]:
sizes = bongo.groupby('template_id').size().sort_values(ascending=False)
print('templates:', len(sizes), '| with 2+ members:', (sizes > 1).sum())
big = sizes.head(8).index
bongo[bongo.template_id.isin(big)].groupby('template_id').agg(n=('id', 'size'), split=('split_template', 'first'), example=('text', 'first')).sort_values('n', ascending=False)

templates: 909 | with 2+ members: 73


,n,split,example
template_id,,,
374,31,test,Habari za asubuhi. Mimi mwenye nyumba wako hii namba yangu ya Vodacom. Mbona kimya na siku zinazidi kwenda...?
62,14,val,IYO PESA ITUME KWENYE NAMBA HII <PHONE> JINA ITALETA Magomba Maila NIMEAZIMA SIMU KWA WAKALA HAPA LAINI YANGU NATATIZO UPANDE WA KUPOKEA...
55,9,train,HABARI ZAUKO NI MIMI MWENYE NYUMBA WAKO MNA ENDEREAJE APO NYUMBANI IYO NAMBA YANGU MPYA YA Halotel NAKUMBUSHIA KODI YANGU MANA NAONA KIM...
189,9,train,"Imethibitishwa, namba yako ya <PHONE> imejishindia <AMOUNT> kutoka TUZO POINTI. Piga <PHONE> ili kupokea Pesa yako."
113,5,train,mjukuu wangu utafuta ji wako mgumu Pesa hazikai mkononi pakazinaisha unasota sana mpenzi hamuelewani je utatunza siri nikikusaidia pesa ...
560,5,train,Tumia namba hii ya Airtel <PHONE> kutuma hiyo ela jina Magomba Maila
681,5,train,"Habari za siku,mimi OLIVA MATIAS,hii namba yangu ya VODACOM.Vp mbona shem OLIVA MATIAS, anakupigia hupatikan tatizo mtandao nn?basi mkiw..."
201,4,train,"Habari za mda huu,mimi Magomba Maila,hii namba yangu ya vodacom.Vp mbona shem Magomba Maila, anakupigia hupatikan tatizo mtandao nn?basi..."


In [7]:
pd.DataFrame({k: stats[k] for k in ('split_random', 'split_template')}).T

,test,train,val
split_random,"{'not scam': 75, 'scam': 85}","{'not scam': 347, 'scam': 397}","{'not scam': 75, 'scam': 85}"
split_template,"{'not scam': 71, 'scam': 81}","{'not scam': 355, 'scam': 405}","{'not scam': 71, 'scam': 81}"


Templates crossing parts of each split (0 means no leakage):

In [8]:
{k: stats[k] for k in ('split_random_templates_crossing_parts', 'split_template_templates_crossing_parts')}

{'split_random_templates_crossing_parts': 40,
 'split_template_templates_crossing_parts': 0}

## The shortcut: genuine BongoScam messages never contain a number

The genuine class is casual personal chat. So *whether a message contains a phone number* almost decides the label, in Swahili but not in Chichewa, where genuine texts include telco service messages.

In [9]:
chi = load_chichewa()
rows = []
for name, df in [('BongoScam', bongo), ('Chichewa', chi)]:
    for p in ['<PHONE>', '<AMOUNT>', '<URL>']:
        r = df.groupby('label').text.apply(lambda s: s.str.contains(p, regex=False).mean())
        rows.append({'data': name, 'placeholder': p, 'share of genuine': r[0], 'share of scams': r[1]})
pd.DataFrame(rows).round(3)

,data,placeholder,share of genuine,share of scams
0,BongoScam,<PHONE>,0.000,0.854
1,BongoScam,<AMOUNT>,0.002,0.081
2,BongoScam,<URL>,0.000,0.005
3,Chichewa,<PHONE>,0.030,0.557
4,Chichewa,<AMOUNT>,0.144,0.149
5,Chichewa,<URL>,0.040,0.003


## Chichewa set (sealed until RQ3)

In [10]:
print(chi.groupby(['source', 'label']).size())
chi.sample(5, random_state=1)[['source', 'label', 'text']]

source        label
D_CHI         0        338
              1        336
telcoSMS_CHI  0         59
dtype: int64


,source,label,text
584,D_CHI,0,ndine Anthony mzako wakale
262,D_CHI,1,"MIRACLE MONEY(Satanism) anthu 200 oyambilira kutijowina kuno ku satanic adzalandila 2 million kwacha, khalani oyambilira poyimba 4n ku (..."
159,D_CHI,0,ndine john ndafika
402,D_CHI,0,gwiritsani ntchito njira zakulela kuti musatenge mimba mosakonzekela.
521,D_CHI,1,UNDUNA WA ZAUMOYO:Ndife ogwira ntchito ku unduna wa zaumoyo. Mupemphedwa kutumiza ndalama kunambala nambala iyi <PHONE> kuti mmulandile ...
